# 🧠 Mental Health Platform — Data Analysis
**DA Intern Task | Full Pipeline: Data Cleaning → SQL → Insights → Features**

---
**Dataset columns expected:**
- `user_id`, `session_id`, `therapist_id`, `session_number`, `session_date`
- `session_status` (completed / cancelled / no-show)
- `rating` (1–5), `revenue`, `acquisition_source`

Adjust column names in **Section 1** if yours differ.

## 0. Install & Import

In [ ]:
# ── Install
!pip install pandas numpy matplotlib seaborn scipy plotly openpyxl --quiet

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats
import warnings, io, os
warnings.filterwarnings('ignore')

# Plotting style
plt.rcParams.update({
    'figure.dpi': 130,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.family': 'DejaVu Sans',
    'axes.titlesize': 13,
    'axes.labelsize': 11,
})
PALETTE = ['#4361ee', '#3a0ca3', '#7209b7', '#f72585', '#4cc9f0']
print('✅ Libraries ready')

## 1. Load Data
Run **one** of the two cells below depending on how you provide the data.

In [ ]:
# ── Option A: Load directly from Google Sheets (make sure sheet is set to 'Anyone with link can view')
SHEET_ID = '1gPX0RSgrer8XeWmu9CyOksTISFMByXhs9pOP44Spbrc'
url = f'https://docs.google.com/spreadsheets/d/{SHEET_ID}/export?format=csv'
df_raw = pd.read_csv(url)
print(f'✅ Loaded {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns')
df_raw.head(3)

In [ ]:
# ── Option B: Upload a CSV/Excel file manually
# from google.colab import files
# uploaded = files.upload()
# fname = list(uploaded.keys())[0]
# df_raw = pd.read_csv(io.BytesIO(uploaded[fname])) if fname.endswith('.csv') \
#          else pd.read_excel(io.BytesIO(uploaded[fname]))
# print(f'✅ Loaded {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns')

## 2. Column Mapping & Quick Exploration

In [ ]:
# ── Standardise column names to snake_case
df_raw.columns = (
    df_raw.columns
    .str.strip()
    .str.lower()
    .str.replace(r'[\s\-]+', '_', regex=True)
)

print('Columns found:', df_raw.columns.tolist())
print('\nShape:', df_raw.shape)
df_raw.info()

In [ ]:
# ── Map to canonical names used throughout this notebook
# Edit the RIGHT side to match YOUR actual column names
COL_MAP = {
    'user_id'          : 'user_id',
    'session_id'       : 'session_id',
    'therapist_id'     : 'therapist_id',
    'session_number'   : 'session_number',
    'session_date'     : 'session_date',
    'session_status'   : 'session_status',
    'rating'           : 'rating',
    'revenue'          : 'revenue',
    'acquisition_source': 'acquisition_source',
}

# Rename only cols that exist
rename_map = {v: k for k, v in COL_MAP.items() if v in df_raw.columns}
df_raw = df_raw.rename(columns=rename_map)

# Keep only canonical columns that exist
KEEP = [c for c in COL_MAP if c in df_raw.columns]
df = df_raw[KEEP].copy()
print('Working columns:', df.columns.tolist())

## 3. Python — Data Cleaning & Validation

### 3.1 Handle Missing Values

In [ ]:
print('=== Missing Values (before cleaning) ===')
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'missing_count': missing, 'missing_%': missing_pct})
print(missing_df[missing_df.missing_count > 0].to_string())

# Strategy:
# - rating: median imputation (ordinal, subject to skew)
# - revenue: median imputation per session_number
# - session_status: mode imputation
# - acquisition_source: fill with 'Unknown'
# - Critical IDs with nulls: drop rows

CRITICAL_COLS = [c for c in ['user_id', 'session_id', 'session_date', 'session_number'] if c in df.columns]
before = len(df)
df = df.dropna(subset=CRITICAL_COLS)
print(f'\nDropped {before - len(df)} rows with null critical identifiers')

if 'rating' in df.columns:
    med_rating = df['rating'].median()
    df['rating'] = df['rating'].fillna(med_rating)
    print(f'rating: filled nulls with median = {med_rating}')

if 'revenue' in df.columns:
    if 'session_number' in df.columns:
        df['revenue'] = df.groupby('session_number')['revenue'].transform(lambda x: x.fillna(x.median()))
    df['revenue'] = df['revenue'].fillna(df['revenue'].median())
    print(f'revenue: filled nulls with session-level median')

if 'session_status' in df.columns:
    mode_status = df['session_status'].mode()[0]
    df['session_status'] = df['session_status'].fillna(mode_status)
    print(f'session_status: filled nulls with mode = {mode_status}')

if 'acquisition_source' in df.columns:
    df['acquisition_source'] = df['acquisition_source'].fillna('Unknown')

print('\n=== Missing Values (after cleaning) ===')
print(df.isnull().sum())

### 3.2 Remove Duplicates

In [ ]:
before = len(df)

# Full-row duplicates
full_dups = df.duplicated().sum()
df = df.drop_duplicates()
print(f'Full-row duplicates removed: {full_dups}')

# Duplicate (user_id, session_number) — keep first occurrence
if 'user_id' in df.columns and 'session_number' in df.columns:
    key_dups = df.duplicated(subset=['user_id', 'session_number']).sum()
    df = df.drop_duplicates(subset=['user_id', 'session_number'], keep='first')
    print(f'(user_id, session_number) duplicates removed: {key_dups}')

print(f'Rows after deduplication: {len(df):,}  (removed {before - len(df)} total)')

### 3.3 Type Conversions & Validation

In [ ]:
# Date parsing
if 'session_date' in df.columns:
    df['session_date'] = pd.to_datetime(df['session_date'], infer_datetime_format=True, errors='coerce')
    bad_dates = df['session_date'].isna().sum()
    if bad_dates:
        print(f'⚠️  {bad_dates} rows with unparseable dates — dropping')
        df = df.dropna(subset=['session_date'])
    df['year_month'] = df['session_date'].dt.to_period('M').astype(str)
    print('✅ session_date parsed')

# Numeric coercions
for col in ['session_number', 'rating', 'revenue']:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

# Rating range guard (1–5)
if 'rating' in df.columns:
    out_of_range = ((df['rating'] < 1) | (df['rating'] > 5)).sum()
    df.loc[(df['rating'] < 1) | (df['rating'] > 5), 'rating'] = np.nan
    df['rating'] = df['rating'].fillna(df['rating'].median())
    print(f'⚠️  {out_of_range} out-of-range ratings corrected to median')

# Revenue non-negative
if 'revenue' in df.columns:
    neg_rev = (df['revenue'] < 0).sum()
    df.loc[df['revenue'] < 0, 'revenue'] = np.nan
    df['revenue'] = df['revenue'].fillna(df['revenue'].median())
    print(f'⚠️  {neg_rev} negative revenue values corrected')

print('\nFinal dtypes:')
print(df.dtypes)

### 3.4 Validate session_number Sequence

In [ ]:
if 'session_number' in df.columns and 'session_date' in df.columns:
    df_sorted = df.sort_values(['user_id', 'session_date'])
    df_sorted['expected_session_number'] = df_sorted.groupby('user_id').cumcount() + 1

    mismatches = (df_sorted['session_number'] != df_sorted['expected_session_number']).sum()
    print(f'Session number mismatches found: {mismatches}')

    if mismatches > 0:
        print('Sample mismatches:')
        mask = df_sorted['session_number'] != df_sorted['expected_session_number']
        print(df_sorted[mask][['user_id','session_date','session_number','expected_session_number']].head(10))
        # Fix: use recalculated sequence
        df_sorted['session_number'] = df_sorted['expected_session_number']
        df = df_sorted.drop(columns='expected_session_number')
        print('✅ session_number corrected to chronological sequence')
    else:
        df = df_sorted.drop(columns='expected_session_number')
        print('✅ session_number sequence is valid')

print(f'\nClean dataset: {len(df):,} rows')

### 3.5 Feature Engineering

In [ ]:
# ── user_lifetime_sessions
if 'user_id' in df.columns:
    lifetime = df.groupby('user_id')['session_number'].max().rename('user_lifetime_sessions')
    df = df.merge(lifetime, on='user_id', how='left')
    print('✅ user_lifetime_sessions created')

# ── avg_rating per user
if 'rating' in df.columns:
    avg_r = df.groupby('user_id')['rating'].mean().round(2).rename('avg_rating')
    df = df.merge(avg_r, on='user_id', how='left')
    print('✅ avg_rating created')

# ── days_between_sessions
if 'session_date' in df.columns:
    df = df.sort_values(['user_id', 'session_date'])
    df['days_between_sessions'] = (
        df.groupby('user_id')['session_date']
        .diff()
        .dt.days
    )
    print('✅ days_between_sessions created')

df.head(5)

### 3.6 Average Rating per Therapist

In [ ]:
if 'therapist_id' in df.columns and 'rating' in df.columns:
    therapist_rating = (
        df.groupby('therapist_id')['rating']
        .agg(avg_rating='mean', session_count='count')
        .reset_index()
        .sort_values('avg_rating', ascending=False)
        .round({'avg_rating': 2})
    )
    print('=== Average Rating per Therapist ===')
    print(therapist_rating.to_string(index=False))

    fig, ax = plt.subplots(figsize=(10, 4))
    bars = ax.barh(
        therapist_rating['therapist_id'].astype(str),
        therapist_rating['avg_rating'],
        color=PALETTE[0], edgecolor='none'
    )
    ax.axvline(therapist_rating['avg_rating'].mean(), color='#f72585',
               linestyle='--', linewidth=1.2, label='Platform avg')
    ax.set_xlabel('Average Rating')
    ax.set_title('Average Rating per Therapist')
    ax.legend(fontsize=9)
    ax.set_xlim(0, 5.2)
    for bar in bars:
        ax.text(bar.get_width() + 0.05, bar.get_y() + bar.get_height()/2,
                f'{bar.get_width():.2f}', va='center', fontsize=9)
    plt.tight_layout()
    plt.savefig('therapist_ratings.png', bbox_inches='tight')
    plt.show()

### 3.7 Correlation: Rating vs Retention

In [ ]:
if 'user_lifetime_sessions' in df.columns and 'avg_rating' in df.columns:
    user_level = df.drop_duplicates('user_id')[['user_id','avg_rating','user_lifetime_sessions']].dropna()

    pearson_r, pearson_p = stats.pearsonr(user_level['avg_rating'], user_level['user_lifetime_sessions'])
    spearman_r, spearman_p = stats.spearmanr(user_level['avg_rating'], user_level['user_lifetime_sessions'])

    print(f'Pearson  r = {pearson_r:.3f},  p = {pearson_p:.4f}')
    print(f'Spearman r = {spearman_r:.3f},  p = {spearman_p:.4f}')

    if abs(pearson_r) > 0.3 and pearson_p < 0.05:
        direction = 'positive' if pearson_r > 0 else 'negative'
        print(f'\n📌 Statistically significant {direction} correlation: higher ratings → {'more' if pearson_r > 0 else 'fewer'} sessions attended.')
    else:
        print('\n📌 Correlation not statistically significant at α=0.05 — rating alone does not explain retention.')

    fig, ax = plt.subplots(figsize=(7, 5))
    ax.scatter(user_level['avg_rating'], user_level['user_lifetime_sessions'],
               alpha=0.45, color=PALETTE[0], edgecolors='none', s=40)
    m, b = np.polyfit(user_level['avg_rating'], user_level['user_lifetime_sessions'], 1)
    x_line = np.linspace(user_level['avg_rating'].min(), user_level['avg_rating'].max(), 100)
    ax.plot(x_line, m*x_line + b, color='#f72585', linewidth=1.5, label=f'Trend  r={pearson_r:.2f}')
    ax.set_xlabel('Average Rating (per user)')
    ax.set_ylabel('Total Sessions Attended')
    ax.set_title('Rating vs Retention')
    ax.legend(fontsize=9)
    plt.tight_layout()
    plt.savefig('rating_vs_retention.png', bbox_inches='tight')
    plt.show()

---
## 4. SQL Queries (via pandas — equivalent logic)
These queries are written as SQL *and* replicated with pandas so you can run them directly in Colab. Copy the SQL blocks into BigQuery / any SQL engine connected to your data.

### SQL 1 — Cohort by Month of First Session

In [ ]:
SQL_COHORT = """
-- SQL 1: Cohort — month of first session per user
SELECT
    user_id,
    DATE_TRUNC(MIN(session_date), MONTH) AS cohort_month
FROM sessions
GROUP BY user_id;
"""
print(SQL_COHORT)

# ── pandas equivalent
cohort = (
    df.groupby('user_id')['session_date']
    .min()
    .dt.to_period('M')
    .reset_index()
    .rename(columns={'session_date': 'cohort_month'})
)
print(cohort.head(5).to_string(index=False))
print(f'\nTotal cohorts: {cohort["cohort_month"].nunique()}')

### SQL 2 — Retention % for Sessions 2, 3, 4

In [ ]:
SQL_RETENTION = """
-- SQL 2: Retention rate for session 2, 3, 4 per cohort
WITH cohort_base AS (
    SELECT user_id, DATE_TRUNC(MIN(session_date), MONTH) AS cohort_month
    FROM sessions GROUP BY user_id
),
session_counts AS (
    SELECT c.cohort_month,
           COUNT(DISTINCT c.user_id)                                              AS cohort_size,
           COUNT(DISTINCT CASE WHEN s.session_number = 2 THEN s.user_id END)      AS ret_s2,
           COUNT(DISTINCT CASE WHEN s.session_number = 3 THEN s.user_id END)      AS ret_s3,
           COUNT(DISTINCT CASE WHEN s.session_number = 4 THEN s.user_id END)      AS ret_s4
    FROM cohort_base c
    JOIN sessions s USING(user_id)
    GROUP BY c.cohort_month
)
SELECT cohort_month,
       cohort_size,
       ROUND(ret_s2 / cohort_size * 100, 1) AS pct_session2,
       ROUND(ret_s3 / cohort_size * 100, 1) AS pct_session3,
       ROUND(ret_s4 / cohort_size * 100, 1) AS pct_session4
FROM session_counts
ORDER BY cohort_month;
"""
print(SQL_RETENTION)

# ── pandas equivalent
df_c = df.merge(cohort, on='user_id')

cohort_size = cohort.groupby('cohort_month')['user_id'].nunique().rename('cohort_size')

retention_rows = []
for sn in [2, 3, 4]:
    had_session = df_c[df_c['session_number'] == sn].groupby('cohort_month')['user_id'].nunique()
    retention_rows.append(had_session.rename(f'users_s{sn}'))

retention = pd.concat([cohort_size] + retention_rows, axis=1).fillna(0)
for sn in [2, 3, 4]:
    retention[f'pct_s{sn}'] = (retention[f'users_s{sn}'] / retention['cohort_size'] * 100).round(1)

retention = retention.reset_index()
print(retention.to_string(index=False))

In [ ]:
# ── Retention curve chart
fig, ax = plt.subplots(figsize=(10, 5))
for i, col in enumerate(['pct_s2', 'pct_s3', 'pct_s4']):
    ax.plot(retention['cohort_month'].astype(str), retention[col],
            marker='o', label=f'Session {i+2}', color=PALETTE[i], linewidth=2)

ax.set_xlabel('Cohort Month')
ax.set_ylabel('Retention (%)')
ax.set_title('Cohort Retention Rate — Sessions 2, 3, 4')
ax.legend()
ax.set_ylim(0, 100)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('cohort_retention.png', bbox_inches='tight')
plt.show()

### SQL 3 — Conversion Funnel (S1 → S2 → S3)

In [ ]:
SQL_FUNNEL = """
-- SQL 3: Conversion funnel
SELECT
    COUNT(DISTINCT user_id)                                              AS total_users,
    COUNT(DISTINCT CASE WHEN session_number >= 1 THEN user_id END)      AS completed_s1,
    COUNT(DISTINCT CASE WHEN session_number >= 2 THEN user_id END)      AS completed_s2,
    COUNT(DISTINCT CASE WHEN session_number >= 3 THEN user_id END)      AS completed_s3,
    ROUND(COUNT(DISTINCT CASE WHEN session_number >= 1 THEN user_id END)
          / COUNT(DISTINCT user_id) * 100, 1)                           AS conv_s1_pct,
    ROUND(COUNT(DISTINCT CASE WHEN session_number >= 2 THEN user_id END)
          / COUNT(DISTINCT CASE WHEN session_number >= 1 THEN user_id END) * 100, 1) AS conv_s2_pct,
    ROUND(COUNT(DISTINCT CASE WHEN session_number >= 3 THEN user_id END)
          / COUNT(DISTINCT CASE WHEN session_number >= 2 THEN user_id END) * 100, 1) AS conv_s3_pct
FROM sessions;
"""
print(SQL_FUNNEL)

# ── pandas equivalent
total_users = df['user_id'].nunique()
s1_users = df[df['session_number'] >= 1]['user_id'].nunique()
s2_users = df[df['session_number'] >= 2]['user_id'].nunique()
s3_users = df[df['session_number'] >= 3]['user_id'].nunique()

funnel = pd.DataFrame({
    'stage'      : ['All Users', 'Completed S1', 'Completed S2', 'Completed S3'],
    'users'      : [total_users, s1_users, s2_users, s3_users],
    'conv_from_prev': [
        100,
        round(s1_users / total_users * 100, 1),
        round(s2_users / s1_users * 100, 1) if s1_users else 0,
        round(s3_users / s2_users * 100, 1) if s2_users else 0,
    ]
})
print(funnel.to_string(index=False))

# ── Funnel chart
fig, ax = plt.subplots(figsize=(8, 5))
colors = [PALETTE[0], PALETTE[1], PALETTE[2], PALETTE[3]]
bars = ax.barh(funnel['stage'][::-1], funnel['users'][::-1], color=colors, edgecolor='none')
for bar, (_, row) in zip(bars, funnel[::-1].iterrows()):
    ax.text(bar.get_width() + total_users * 0.01, bar.get_y() + bar.get_height()/2,
            f"{row['users']:,}  ({row['conv_from_prev']}%)", va='center', fontsize=10)
ax.set_xlabel('Users')
ax.set_title('Conversion Funnel')
ax.set_xlim(0, total_users * 1.25)
plt.tight_layout()
plt.savefig('funnel.png', bbox_inches='tight')
plt.show()

### SQL 4 — Revenue Metrics

In [ ]:
SQL_REVENUE = """
-- SQL 4a: Total revenue & revenue per user
SELECT
    ROUND(SUM(revenue), 2)                       AS total_revenue,
    ROUND(SUM(revenue) / COUNT(DISTINCT user_id), 2) AS revenue_per_user
FROM sessions;

-- SQL 4b: Revenue per cohort
WITH cohort_base AS (
    SELECT user_id, DATE_TRUNC(MIN(session_date), MONTH) AS cohort_month
    FROM sessions GROUP BY user_id
)
SELECT c.cohort_month,
       ROUND(SUM(s.revenue), 2)                          AS cohort_revenue,
       COUNT(DISTINCT c.user_id)                          AS cohort_users,
       ROUND(SUM(s.revenue)/COUNT(DISTINCT c.user_id), 2) AS revenue_per_user
FROM cohort_base c JOIN sessions s USING(user_id)
GROUP BY c.cohort_month ORDER BY c.cohort_month;

-- SQL 4c: Revenue per therapist
SELECT therapist_id,
       COUNT(*) AS sessions_completed,
       ROUND(SUM(revenue), 2) AS total_revenue,
       ROUND(AVG(revenue), 2) AS avg_revenue_per_session
FROM sessions
WHERE session_status = 'completed'
GROUP BY therapist_id
ORDER BY total_revenue DESC;
"""
print(SQL_REVENUE)

# ── pandas equivalents
if 'revenue' in df.columns:
    completed = df[df['session_status'].str.lower() == 'completed'] if 'session_status' in df.columns else df

    total_rev = completed['revenue'].sum()
    rev_per_user = total_rev / completed['user_id'].nunique()
    print(f'\n📊 Total Revenue: ${total_rev:,.2f}')
    print(f'📊 Revenue per User: ${rev_per_user:,.2f}')

    # Revenue per cohort
    cohort_rev = (
        completed
        .merge(cohort, on='user_id')
        .groupby('cohort_month')
        .agg(cohort_revenue=('revenue','sum'), cohort_users=('user_id','nunique'))
        .assign(rev_per_user=lambda x: (x.cohort_revenue / x.cohort_users).round(2))
        .reset_index()
    )
    print('\n=== Revenue per Cohort ===')
    print(cohort_rev.to_string(index=False))

    # Revenue per therapist
    if 'therapist_id' in df.columns:
        therapist_rev = (
            completed
            .groupby('therapist_id')
            .agg(sessions=('session_id','count'), total_revenue=('revenue','sum'))
            .assign(avg_per_session=lambda x: (x.total_revenue / x.sessions).round(2))
            .sort_values('total_revenue', ascending=False)
            .reset_index()
        )
        print('\n=== Revenue per Therapist ===')
        print(therapist_rev.to_string(index=False))

### SQL 4e — Monthly Therapist Bonus (>10 sessions → 20% bonus)

In [ ]:
SQL_BONUS = """
-- SQL 4e: Monthly bonus — therapists completing >10 sessions get 20% of their monthly revenue
WITH monthly_stats AS (
    SELECT
        therapist_id,
        DATE_TRUNC(session_date, MONTH)   AS month,
        COUNT(*)                           AS sessions_in_month,
        SUM(revenue)                       AS monthly_revenue
    FROM sessions
    WHERE session_status = 'completed'
    GROUP BY therapist_id, DATE_TRUNC(session_date, MONTH)
)
SELECT
    therapist_id,
    month,
    sessions_in_month,
    ROUND(monthly_revenue, 2)              AS monthly_revenue,
    ROUND(monthly_revenue * 0.20, 2)       AS bonus_amount
FROM monthly_stats
WHERE sessions_in_month > 10
ORDER BY month, therapist_id;
"""
print(SQL_BONUS)

# ── pandas equivalent
if 'revenue' in df.columns and 'therapist_id' in df.columns:
    comp = df[df['session_status'].str.lower() == 'completed'].copy() if 'session_status' in df.columns else df.copy()
    comp['month'] = comp['session_date'].dt.to_period('M')

    monthly_stats = (
        comp.groupby(['therapist_id', 'month'])
        .agg(sessions_in_month=('session_id','count'), monthly_revenue=('revenue','sum'))
        .reset_index()
    )
    bonus = monthly_stats[monthly_stats['sessions_in_month'] > 10].copy()
    bonus['bonus_amount'] = (bonus['monthly_revenue'] * 0.20).round(2)

    print(f'\nMonths where bonus is triggered: {len(bonus)}')
    print(bonus.sort_values('month').to_string(index=False))

    total_bonus = bonus['bonus_amount'].sum()
    print(f'\n💰 Total bonus outlay: ${total_bonus:,.2f}')

### SQL 5 — Drop-off Distribution (at which session do users leave?)

In [ ]:
SQL_DROPOFF = """
-- SQL 5: Session at which users drop off (last session per user)
WITH last_session AS (
    SELECT user_id, MAX(session_number) AS last_session_number
    FROM sessions
    GROUP BY user_id
)
SELECT
    last_session_number               AS dropped_after_session,
    COUNT(*)                           AS users_dropped,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 1) AS pct_of_dropoffs
FROM last_session
GROUP BY last_session_number
ORDER BY last_session_number;
"""
print(SQL_DROPOFF)

# ── pandas equivalent
last_sess = df.groupby('user_id')['session_number'].max().reset_index()
last_sess.columns = ['user_id', 'dropped_after_session']

dropoff = (
    last_sess.groupby('dropped_after_session')['user_id']
    .count()
    .reset_index()
    .rename(columns={'user_id': 'users_dropped'})
)
dropoff['pct'] = (dropoff['users_dropped'] / dropoff['users_dropped'].sum() * 100).round(1)
print(dropoff.to_string(index=False))
peak_dropoff = dropoff.loc[dropoff['users_dropped'].idxmax()]
print(f'\n🔴 Highest drop-off: after session {int(peak_dropoff["dropped_after_session"])} — {peak_dropoff["pct"]}% of all churned users')

# ── Drop-off chart
fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(dropoff['dropped_after_session'].astype(str), dropoff['users_dropped'],
       color=PALETTE[3], edgecolor='none')
for _, row in dropoff.iterrows():
    ax.text(str(row['dropped_after_session']), row['users_dropped'] + 0.5,
            f"{row['pct']}%", ha='center', fontsize=9)
ax.set_xlabel('Session Number (last session attended)')
ax.set_ylabel('Users who dropped off')
ax.set_title('Drop-off Distribution by Session')
plt.tight_layout()
plt.savefig('dropoff.png', bbox_inches='tight')
plt.show()

## 5. Revenue Trend Over Time

In [ ]:
if 'revenue' in df.columns:
    rev_trend = (
        df.groupby('year_month')['revenue'].sum()
        .reset_index()
        .sort_values('year_month')
    )

    fig, ax = plt.subplots(figsize=(11, 4))
    ax.fill_between(rev_trend['year_month'], rev_trend['revenue'], alpha=0.18, color=PALETTE[0])
    ax.plot(rev_trend['year_month'], rev_trend['revenue'], color=PALETTE[0], linewidth=2, marker='o', markersize=4)
    ax.set_xlabel('Month')
    ax.set_ylabel('Revenue ($)')
    ax.set_title('Monthly Revenue Trend')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.savefig('revenue_trend.png', bbox_inches='tight')
    plt.show()

## 6. Acquisition Source Analysis

In [ ]:
if 'acquisition_source' in df.columns:
    acq = (
        df.drop_duplicates('user_id')
        .merge(last_sess, on='user_id')
        .groupby('acquisition_source')
        .agg(
            users=('user_id','count'),
            avg_lifetime_sessions=('dropped_after_session','mean'),
            avg_rating=('avg_rating','mean') if 'avg_rating' in df.columns else ('user_id','count'),
        )
        .round(2)
        .sort_values('avg_lifetime_sessions', ascending=False)
        .reset_index()
    )
    if 'revenue' in df.columns:
        acq_rev = (
            df.merge(df.drop_duplicates('user_id')[['user_id','acquisition_source']], on='user_id')
            .groupby('acquisition_source')['revenue'].sum().round(2).reset_index()
        )
        acq = acq.merge(acq_rev, on='acquisition_source')

    print('=== Acquisition Source Performance ===')
    print(acq.to_string(index=False))

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].barh(acq['acquisition_source'], acq['users'], color=PALETTE[0], edgecolor='none')
    axes[0].set_title('Users by Acquisition Source')
    axes[1].barh(acq['acquisition_source'], acq['avg_lifetime_sessions'], color=PALETTE[2], edgecolor='none')
    axes[1].set_title('Avg Lifetime Sessions by Source')
    plt.tight_layout()
    plt.savefig('acquisition.png', bbox_inches='tight')
    plt.show()

## 7. Export Clean Data (for Looker Studio)

In [ ]:
# Export main clean file
df.to_csv('sessions_clean.csv', index=False)

# Export summary tables
retention.to_csv('cohort_retention.csv', index=False)
dropoff.to_csv('dropoff_distribution.csv', index=False)

if 'revenue' in df.columns:
    rev_trend.to_csv('revenue_trend.csv', index=False)
    cohort_rev.to_csv('revenue_per_cohort.csv', index=False)
    therapist_rev.to_csv('revenue_per_therapist.csv', index=False)
    bonus.to_csv('therapist_bonus.csv', index=False)

if 'acquisition_source' in df.columns:
    acq.to_csv('acquisition_analysis.csv', index=False)

funnel.to_csv('conversion_funnel.csv', index=False)

print('✅ All CSVs exported — upload to Google Drive to connect to Looker Studio')

# Optional: zip everything
import zipfile
with zipfile.ZipFile('looker_studio_data.zip', 'w') as zf:
    for fname in [
        'sessions_clean.csv','cohort_retention.csv','dropoff_distribution.csv',
        'revenue_trend.csv','revenue_per_cohort.csv','revenue_per_therapist.csv',
        'therapist_bonus.csv','acquisition_analysis.csv','conversion_funnel.csv'
    ]:
        if os.path.exists(fname):
            zf.write(fname)
print('📦 looker_studio_data.zip ready for download')

---
## 8. Insights & Recommendations

### 8.1 Why are users dropping off?
Based on the drop-off distribution analysis:
- **Session 1 → 2 is the most critical churn point** for most subscription-based therapy platforms. Users who attend only one session often did so out of curiosity or urgency — without strong onboarding or a clear next-step, they don't return.
- **Possible causes:** mismatch between user expectations and the therapy format; poor therapist–client fit; perceived high cost; friction in rebooking; lack of reminder/nudge communication.
- **Data signal to check:** Compare `avg_rating` for users who churned after session 1 vs those who continued. If ratings are similar, the issue is operational (rebooking friction), not satisfaction.

### 8.2 Which acquisition source performs best?
- Rank sources by **avg_lifetime_sessions** (not just user volume). A high-volume channel with 1.2 avg sessions is worse than a smaller channel with 4.5 avg sessions.
- Cross-reference with revenue per user per source to calculate **Customer Lifetime Value (CLV) by channel**.
- Recommendation: **Shift budget toward the highest CLV source**, even if volume is lower. Organic/referral sources typically produce higher-retention users in mental health contexts.

### 8.3 How can we improve retention?
1. **Automated re-engagement at 48h post-session 1:** Send a personalised message referencing the therapist by name and suggesting session 2 scheduling.
2. **Therapist–client matching:** Users matched on communication style (CBT vs talk therapy) show higher continuity. Surface matching attributes at signup.
3. **Milestone acknowledgement:** At session 3, send a progress summary. Research shows perceived progress is a top predictor of continued therapy.
4. **Flexible pricing / trial packs:** Offer a 3-session starter pack at a discount — reduces friction at the 2nd booking decision point.
5. **Flag low-rating sessions immediately:** Any session rated ≤2 should trigger an outreach from a care coordinator within 24h.

### 8.4 What metric would you track daily?
**Primary daily metric: Session 2 Booking Rate (S2BR)**
- Definition: `% of users who completed session 1 in the last 7 days who have booked session 2`
- Why: It is the leading indicator of retention — it precedes and predicts all downstream revenue and engagement. It is actionable (teams can intervene same-day), and it updates frequently enough to catch problems before they compound.
- **Supporting daily metrics:**
  - Daily completed sessions (volume health check)
  - Average rating of sessions completed yesterday (quality signal)
  - Cancellation/no-show rate (operational efficiency)

---
## 9. Looker Studio Dashboard — Build Guide

Upload the CSV files exported in Section 7 to Google Drive, then connect each as a data source in Looker Studio.

**Recommended page layout (3 pages):**

**Page 1 — Executive Summary**
- Scorecards: Total Revenue | Revenue per User | Total Users | Avg Rating
- Time series: Revenue Trend (revenue_trend.csv)
- Bar: Top 5 Therapists by Revenue

**Page 2 — Retention & Funnel**
- Funnel chart: conversion_funnel.csv
- Line chart: Cohort retention curve (sessions 2/3/4) — cohort_retention.csv
- Bar: Drop-off distribution — dropoff_distribution.csv
- Scatter: Rating vs Lifetime Sessions (sessions_clean.csv)

**Page 3 — Acquisition & Therapists**
- Bar: Avg lifetime sessions by acquisition source
- Table: Therapist bonus eligibility — therapist_bonus.csv
- Table: Revenue per cohort — revenue_per_cohort.csv
- Filter controls: Date range, Therapist ID, Acquisition Source